# FlyPose-SAR — Pix2Pix RGB→Thermal (Fase 2b)

## Prerequisiti (Data):
- KAIST preview estratto in `/kaggle/working/kaist-cvpr15-preview/`
- Oppure aggiungi il tar.gz come dataset Kaggle

## Setup: GPU T4 x1 basta (converge in 30 epoche)

## Differenze rispetto alla CycleGAN:
- **Paired**: RGB e thermal sono coppie allineate → supervisione diretta
- **UNet** come generatore (skip connections): preserva dettagli spaziali
- **L1 loss** aggiuntiva: confronto diretto output vs thermal reale
- **Niente ciclo inverso**: un solo generatore, un solo discriminatore
- Converge in 20-30 epoche invece di 200

## Cella 1 — Installazione

In [ ]:
!pip install torch torchvision pillow tqdm -q
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## Cella 2 — Configurazione

In [ ]:
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAIST_ROOT     = KAGGLE_WORKING / 'kaist-cvpr15-preview'

# ---- PARAMETRI ----
IMG_SIZE    = 256
BATCH_SIZE  = 16
EPOCHS      = 30
LR          = 0.0002
BETA1       = 0.5
LAMBDA_L1   = 100.0   # peso L1 loss (standard pix2pix)
SAVE_FREQ   = 5
N_WORKERS   = 4
# Palette colormap da applicare al thermal grayscale
# Opzioni: cv2.COLORMAP_CIVIDIS, COLORMAP_INFERNO, COLORMAP_MAGMA
# Cambia in base alla risposta della regione sul sensore
COLORMAP    = 'CIVIDIS'  # candidato principale dall'analisi MSE
# -------------------

device = 'cuda' if torch.cuda.is_available() else 'cpu'
run_dir = KAGGLE_WORKING / 'pix2pix_run'
run_dir.mkdir(exist_ok=True)

print(f'Device      : {device}')
print(f'KAIST root  : {KAIST_ROOT}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epoche      : {EPOCHS}')
print(f'Colormap    : {COLORMAP}')
print(f'Run dir     : {run_dir}')

## Cella 3 — Dataset KAIST paired

In [ ]:
import cv2
import random
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

COLORMAP_MAP = {
    'CIVIDIS' : cv2.COLORMAP_CIVIDIS,
    'INFERNO' : cv2.COLORMAP_INFERNO,
    'MAGMA'   : cv2.COLORMAP_MAGMA,
    'JET'     : cv2.COLORMAP_JET,
    'NONE'    : None,  # grayscale puro, nessuna palette
}

class KAISTPairedDataset(Dataset):
    """
    Dataset paired KAIST: ogni sample e' una coppia (RGB, thermal colorato).
    Il thermal KAIST e' grayscale LWIR — applichiamo la colormap scelta
    per simulare l'output del sensore della regione.
    """
    def __init__(self, kaist_root, img_size=256, augment=True, colormap='CIVIDIS'):
        self.img_size  = img_size
        self.augment   = augment
        self.cmap_id   = COLORMAP_MAP.get(colormap, cv2.COLORMAP_CIVIDIS)
        self.pairs     = []

        root = Path(kaist_root)
        # Cerca tutte le immagini visible e trova il corrispondente lwir
        for vis_path in sorted(root.rglob('visible/*.jpg')):
            # Salta file macOS nascosti
            if vis_path.name.startswith('._'):
                continue
            lwir_path = vis_path.parent.parent / 'lwir' / vis_path.name
            if lwir_path.exists():
                self.pairs.append((vis_path, lwir_path))

        print(f'[Dataset] Coppie trovate: {len(self.pairs)}')
        if self.pairs:
            print(f'  Esempio: {self.pairs[0][0].name}')

    def __len__(self):
        return len(self.pairs)

    def _apply_colormap(self, lwir_path):
        """Carica thermal grayscale e applica colormap."""
        gray = cv2.imread(str(lwir_path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            return None
        if self.cmap_id is not None:
            colored = cv2.applyColorMap(gray, self.cmap_id)
            return Image.fromarray(cv2.cvtColor(colored, cv2.COLOR_BGR2RGB))
        else:
            return Image.fromarray(gray).convert('RGB')

    def __getitem__(self, idx):
        vis_path, lwir_path = self.pairs[idx]

        rgb     = Image.open(vis_path).convert('RGB')
        thermal = self._apply_colormap(lwir_path)
        if thermal is None:
            thermal = Image.new('RGB', rgb.size, (0,0,0))

        # Resize
        s = self.img_size
        rgb     = rgb.resize((s, s), Image.BICUBIC)
        thermal = thermal.resize((s, s), Image.BICUBIC)

        # Augmentation sincronizzata su entrambe le immagini
        if self.augment:
            if random.random() > 0.5:
                rgb     = TF.hflip(rgb)
                thermal = TF.hflip(thermal)
            # Jitter solo sull'RGB (il thermal non ha colori da augmentare)
            if random.random() > 0.5:
                rgb = TF.adjust_brightness(rgb, 0.8 + random.random() * 0.4)

        # Tensor normalizzato in [-1, 1]
        to_t = lambda img: TF.normalize(TF.to_tensor(img), (0.5,0.5,0.5), (0.5,0.5,0.5))
        return {'A': to_t(rgb), 'B': to_t(thermal)}


# Crea dataset e dataloader
dataset    = KAISTPairedDataset(KAIST_ROOT, IMG_SIZE, augment=True, colormap=COLORMAP)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=N_WORKERS, pin_memory=True, drop_last=True)
print(f'[OK] {len(dataloader)} batch per epoca')

## Cella 4 — Architettura Pix2Pix (UNet + PatchGAN)

In [ ]:
import torch.nn as nn

# ============================================================
# GENERATORE: UNet con skip connections
# Encoder: comprime l'immagine estraendo feature a scale diverse
# Decoder: ricostruisce l'output usando le feature dell'encoder
# Skip connections: preservano i dettagli spaziali (bordi, strutture)
# ============================================================

class UNetDown(nn.Module):
    """Blocco encoder: Conv + InstanceNorm + LeakyReLU"""
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class UNetUp(nn.Module):
    """Blocco decoder: ConvTranspose + InstanceNorm + ReLU + skip connection"""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        return torch.cat([self.block(x), skip], dim=1)  # skip connection

class UNetGenerator(nn.Module):
    """
    UNet 256x256: 8 livelli encoder + 8 livelli decoder.
    Input:  RGB 3ch 256x256
    Output: thermal 3ch 256x256
    """
    def __init__(self, in_ch=3, out_ch=3, features=64):
        super().__init__()
        # Encoder
        self.d1 = UNetDown(in_ch,       features,   normalize=False)  # 128
        self.d2 = UNetDown(features,    features*2)                    # 64
        self.d3 = UNetDown(features*2,  features*4)                    # 32
        self.d4 = UNetDown(features*4,  features*8)                    # 16
        self.d5 = UNetDown(features*8,  features*8)                    # 8
        self.d6 = UNetDown(features*8,  features*8)                    # 4
        self.d7 = UNetDown(features*8,  features*8)                    # 2
        self.d8 = UNetDown(features*8,  features*8, normalize=False)   # 1 (bottleneck)
        # Decoder (input = output encoder + skip)
        self.u1 = UNetUp(features*8,    features*8, dropout=0.5)       # 2
        self.u2 = UNetUp(features*16,   features*8, dropout=0.5)       # 4
        self.u3 = UNetUp(features*16,   features*8, dropout=0.5)       # 8
        self.u4 = UNetUp(features*16,   features*8)                    # 16
        self.u5 = UNetUp(features*16,   features*4)                    # 32
        self.u6 = UNetUp(features*8,    features*2)                    # 64
        self.u7 = UNetUp(features*4,    features)                      # 128
        self.final = nn.Sequential(
            nn.ConvTranspose2d(features*2, out_ch, 4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)
        d7 = self.d7(d6)
        d8 = self.d8(d7)
        u1 = self.u1(d8, d7)
        u2 = self.u2(u1, d6)
        u3 = self.u3(u2, d5)
        u4 = self.u4(u3, d4)
        u5 = self.u5(u4, d3)
        u6 = self.u6(u5, d2)
        u7 = self.u7(u6, d1)
        return self.final(u7)


# ============================================================
# DISCRIMINATORE: PatchGAN 70x70 (identico alla CycleGAN)
# Input: coppia (RGB, thermal) concatenati → 6 canali
# Il discriminatore vede ENTRAMBE le immagini: giudica se
# la coppia (RGB, thermal) e' reale o generata
# ============================================================

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=6, ndf=64):  # 6 = 3 RGB + 3 thermal
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, ndf, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, stride=1, padding=1),
        )
    def forward(self, rgb, thermal):
        return self.model(torch.cat([rgb, thermal], dim=1))


# Inizializza modelli
def init_weights(net, std=0.02):
    for m in net.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    return net

G = init_weights(UNetGenerator(3, 3)).to(device)
D = init_weights(PatchGANDiscriminator(6)).to(device)

if torch.cuda.device_count() > 1:
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)
    print(f'[OK] DataParallel su {torch.cuda.device_count()} GPU')

# Conta parametri
n_G = sum(p.numel() for p in G.parameters())
n_D = sum(p.numel() for p in D.parameters())
print(f'[OK] G (UNet)    : {n_G/1e6:.1f}M parametri')
print(f'[OK] D (PatchGAN): {n_D/1e6:.1f}M parametri')

## Cella 5 — Ottimizzatori e Loss

In [ ]:
import torch.optim as optim

criterion_GAN = nn.MSELoss()   # LSGAN
criterion_L1  = nn.L1Loss()    # ricostruzione pixel-level

optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

# Scheduler: lr costante per meta' epoche, poi decay lineare a 0
def lr_lambda(epoch):
    if epoch < EPOCHS // 2:
        return 1.0
    return 1.0 - (epoch - EPOCHS // 2) / max(EPOCHS // 2, 1)

scheduler_G = optim.lr_scheduler.LambdaLR(optimizer_G, lr_lambda)
scheduler_D = optim.lr_scheduler.LambdaLR(optimizer_D, lr_lambda)

print('[OK] Ottimizzatori e scheduler pronti')
print(f'     Loss: LSGAN (MSE) + L1 * {LAMBDA_L1}')

## Cella 6 — Training Loop

In [ ]:
import json, time
from pathlib import Path

history = {'epoch': [], 'G': [], 'D': [], 'L1': [], 'adv': []}
MAX_STEPS = 500  # step per epoca (bilancia velocita' e copertura dataset)

print(f'[*] Training Pix2Pix — {EPOCHS} epoche, max {MAX_STEPS} step/epoca')
print(f'    Colormap: {COLORMAP}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    G.train(); D.train()
    g_losses, d_losses, l1_losses, adv_losses = [], [], [], []
    t0 = time.time()

    for step, batch in enumerate(dataloader):
        if step >= MAX_STEPS:
            break

        real_A = batch['A'].to(device)  # RGB
        real_B = batch['B'].to(device)  # thermal colorato reale

        # ── AGGIORNA GENERATORE ──
        optimizer_G.zero_grad()
        fake_B    = G(real_A)
        pred_fake = D(real_A, fake_B)
        loss_adv  = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
        loss_L1   = criterion_L1(fake_B, real_B) * LAMBDA_L1
        loss_G    = loss_adv + loss_L1
        loss_G.backward()
        optimizer_G.step()

        # ── AGGIORNA DISCRIMINATORE ──
        optimizer_D.zero_grad()
        pred_real = D(real_A, real_B)
        pred_fake = D(real_A, fake_B.detach())
        loss_D    = (criterion_GAN(pred_real, torch.ones_like(pred_real)) +
                     criterion_GAN(pred_fake, torch.zeros_like(pred_fake))) * 0.5
        loss_D.backward()
        optimizer_D.step()

        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())
        l1_losses.append(loss_L1.item() / LAMBDA_L1)
        adv_losses.append(loss_adv.item())

        if step % 100 == 0:
            print(f'  step {step:3d}/{MAX_STEPS} | G={loss_G.item():.3f} D={loss_D.item():.3f}')

    scheduler_G.step()
    scheduler_D.step()

    g_mean  = sum(g_losses)  / len(g_losses)
    d_mean  = sum(d_losses)  / len(d_losses)
    l1_mean = sum(l1_losses) / len(l1_losses)
    elapsed = time.time() - t0

    history['epoch'].append(epoch)
    history['G'].append(g_mean)
    history['D'].append(d_mean)
    history['L1'].append(l1_mean)
    history['adv'].append(sum(adv_losses)/len(adv_losses))

    print(f'Epoch {epoch:3d}/{EPOCHS} | G={g_mean:.3f} D={d_mean:.3f} L1={l1_mean:.3f} | {elapsed:.0f}s')

    # Salva checkpoint ogni SAVE_FREQ epoche
    if epoch % SAVE_FREQ == 0:
        g_state = G.module.state_dict() if hasattr(G, 'module') else G.state_dict()
        torch.save(g_state, run_dir / f'G_pix2pix_epoch{epoch:03d}.pth')
        with open(run_dir / 'training_history.json', 'w') as f:
            json.dump(history, f)
        print(f'  [SAVE] checkpoint epoch {epoch}')

# Salva finale
g_state = G.module.state_dict() if hasattr(G, 'module') else G.state_dict()
torch.save(g_state, run_dir / 'G_pix2pix_final.pth')
with open(run_dir / 'training_history.json', 'w') as f:
    json.dump(history, f)
print('[DONE] Training completato!')
print(f'Pesi salvati: {run_dir}/G_pix2pix_final.pth')

## Cella 7 — Visual grid (verifica qualità)

In [ ]:
import matplotlib.pyplot as plt

G.eval()
denorm = lambda t: (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

ROWS = 4
fig, axes = plt.subplots(ROWS, 3, figsize=(12, ROWS * 3.5))

for c, title in enumerate(['RGB Originale', 'Thermal Reale (KAIST)', 'Thermal Generato (Pix2Pix)']):
    axes[0, c].set_title(title, fontweight='bold')

with torch.no_grad():
    for i in range(ROWS):
        sample   = dataset[i * (len(dataset) // ROWS)]
        real_A   = sample['A'].unsqueeze(0).to(device)
        real_B   = sample['B']
        fake_B   = G(real_A)[0]
        for c, img in enumerate([real_A[0], real_B, fake_B]):
            axes[i, c].imshow(denorm(img))
            axes[i, c].axis('off')

plt.suptitle(f'Pix2Pix RGB→Thermal ({COLORMAP}) — Qualità generazione', fontweight='bold')
plt.tight_layout()
plt.savefig(run_dir / 'visual_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('[OK] Grid salvata')

## Cella 8 — Zip e download

In [ ]:
import zipfile

zip_path = KAGGLE_WORKING / f'pix2pix_{COLORMAP.lower()}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(run_dir.rglob('*')):
        if f.is_file():
            zf.write(f, f.relative_to(run_dir.parent))
            print(f'  [+] {f.name}')

print(f'\n[OK] {zip_path} — {zip_path.stat().st_size/1e6:.1f} MB')
print('Scarica da Output (pannello destro)')